# Phase 24: XyetherLineFlowNet v2 (Zero-Halo APISR Geometry + Harsh-Web De-4:2:0 & Shoulder Recovery)
- **100% Own Student Lineage (`XyetherLineFlow_v23_Final.pth` -> `v24`):** Warm-starts from Phase 23 EMA weights (`56c8b`, ~975ms CPU = ~6x faster than `APISR`).
- **Multi-Regime OTF Dataset (`Clean 25% + Medium Web 45% + Harsh Q52..74 Web 30%`):** Eliminates soft mush and reddish 4:2:0 JPEG chroma bleed inside dark fills.
- **Phase 24 Line-Flow Loss + Shallow VGG19 (`0.05` vs `tar_b`):** Adds symmetric bright-shoulder recovery (`l_sh`), teacher-aware bilateral edge barrier, strict-edge `3x3` Laplacian match, and `4x4`/`8x8` block de-chroma.

In [ ]:
# 1. Clone Latest Phase 22 Training Engine & Install GPU ONNX Runtime + Dependencies
import os, sys, torch

!rm -rf /kaggle/working/engine
!git clone https://github.com/Thanawanit/upscale-training.git /kaggle/working/engine
!pip install -q huggingface_hub onnx onnxruntime-gpu onnx2torch scikit-image

sys.path.insert(0, '/kaggle/working/engine')
print(f"✅ Phase 22 Engine cloned! GPUs available: {torch.cuda.device_count()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


In [ ]:
# 2. High-Res Anime Dataset Preparation (Safebooru Absurdres 4K)
import os, requests, cv2
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

DATA_DIR = "/kaggle/working/Elite_Anime_4K"
os.makedirs(DATA_DIR, exist_ok=True)

existing = [f for f in os.listdir(DATA_DIR) if f.endswith(('.png', '.jpg', '.jpeg', '.webp'))]
if len(existing) < 120:
    print("📡 Fetching high-res anime frames from Safebooru...")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    image_urls = []
    for page in range(1, 9):
        try:
            api_url = f"https://safebooru.org/index.php?page=dapi&s=post&q=index&json=1&limit=100&pid={page}&tags=absurdres+rating:safe+-comic+-text"
            resp = requests.get(api_url, headers=headers, timeout=15).json()
            for post in resp:
                img_url = f"https://safebooru.org/images/{post['directory']}/{post['image']}"
                if img_url.endswith(('.png', '.jpg', '.jpeg', '.webp')):
                    image_urls.append(img_url)
        except Exception as e:
            print(f"⚠️ Safebooru Page {page} warning: {e}")

    def dl(url):
        try:
            dest_path = os.path.join(DATA_DIR, url.split('/')[-1])
            if not os.path.exists(dest_path):
                with open(dest_path, 'wb') as f:
                    f.write(requests.get(url, headers=headers, timeout=30).content)
        except Exception:
            pass

    with ThreadPoolExecutor(max_workers=8) as exe:
        list(exe.map(dl, image_urls[:350]))

valid_files = []
for f in Path(DATA_DIR).glob("*.*"):
    if f.suffix.lower() in ['.png', '.jpg', '.jpeg', '.webp']:
        try:
            img = cv2.imread(str(f))
            if img is not None and img.shape[0] >= 256 and img.shape[1] >= 256:
                valid_files.append(f.name)
            else:
                f.unlink()
        except Exception:
            f.unlink()

print(f"✅ Dataset ready: {len(valid_files)} verified anime frames!")


In [ ]:
# 3. Download Authentic Own-Checkpoint Student Base & APISR Teacher ONNX
import os, urllib.request

MODEL_DIR = "/kaggle/working/pretrained_models"
os.makedirs(MODEL_DIR, exist_ok=True)
g_url = "https://huggingface.co/Thanawanit/Kaggle-Backup/resolve/main/checkpoints/Phase23_XyetherLineFlow_v1/XyetherLineFlow_v23_Final.pth"
g_path = os.path.join(MODEL_DIR, "XyetherLineFlow_v23_Final.pth")
t_url = "https://huggingface.co/Thanawanit/Kaggle-Backup/resolve/main/teachers/APISR.onnx"
t_path = os.path.join(MODEL_DIR, "APISR.onnx")

if not os.path.exists(g_path):
    print("Downloading authentic own-checkpoint Phase 23 student (XyetherLineFlow_v23_Final.pth)...")
    urllib.request.urlretrieve(g_url, g_path)
if not os.path.exists(t_path):
    print("Downloading APISR.onnx teacher model...")
    urllib.request.urlretrieve(t_url, t_path)
print(f"Student v23 ({os.path.getsize(g_path)/(1024*1024):.2f} MB) & APISR teacher ({os.path.getsize(t_path)/(1024*1024):.2f} MB) ready!")


In [ ]:
# 4. Run Phase 24 XyetherLineFlowNet Continuation (7500 iters, Hard Web Q52..74, Shoulder + Laplacian + Shallow VGG)
!python /kaggle/working/engine/train_xyether_p23.py \
    --data_dir /kaggle/working/Elite_Anime_4K \
    --resume_path /kaggle/working/pretrained_models/XyetherLineFlow_v23_Final.pth \
    --teacher_onnx_path /kaggle/working/pretrained_models/APISR.onnx \
    --output_dir /kaggle/working/output_models_phase24 \
    --model_prefix XyetherLineFlow_v24 \
    --hf_subdir checkpoints/Phase24_XyetherLineFlow_v2 \
    --loss_version 24 \
    --num_feat 56 \
    --num_blocks 8 \
    --batch_size 12 \
    --patch_size 160 \
    --total_iters 7500 \
    --warmup_iters 200 \
    --lr_g 1.6e-4 \
    --min_lr 5e-6 \
    --vgg_weight 0.05 \
    --luv_weight 0.25 \
    --ema_decay 0.9992 \
    --eval_every 1500 \
    --save_every 2500


In [ ]:
# 5. Automated Cloud GPU Visual, Harsh-Q62, Hairline-Geometry, ONNX-Parity & Cleanup
import os, sys, cv2, json, shutil, urllib.request, requests
import numpy as np
import torch
import torch.nn.functional as F
import onnxruntime as ort
from skimage.metrics import structural_similarity as ssim
from huggingface_hub import HfApi

sys.path.insert(0, '/kaggle/working/engine')
from archs.xyether_compact import XyetherCompactNet, XyetherBalancedNet
from archs.xyether_lineflow import XyetherLineFlowNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('/kaggle/working/eval_models', exist_ok=True)
HF_BASE = 'https://huggingface.co/Thanawanit/Kaggle-Backup/resolve/main/checkpoints'

models_meta = {
    'Balanced_2x': (f'{HF_BASE}/Official_Xyether_References/Xyether_Balanced_2x_Official.pth', 'balanced'),
    'Strong_v3': (f'{HF_BASE}/Official_Xyether_References/Xyether_Strong_v3_Official.pth', 'compact'),
    'Phase22_TwinReflMaster_v3': (f'{HF_BASE}/Phase22_XyetherKiller_APISRMaster_v3/XyetherKiller_APISRMaster_v22_Final.pth', 'compact'),
    'Phase23_LineFlow_v1': (f'{HF_BASE}/Phase23_XyetherLineFlow_v1/XyetherLineFlow_v23_Final.pth', 'lineflow'),
}

def load_net(path, arch_type):
    if arch_type == 'balanced':
        m = XyetherBalancedNet().to(device)
    elif arch_type == 'lineflow':
        m = XyetherLineFlowNet(num_feat=56, num_blocks=8).to(device)
    else:
        m = XyetherCompactNet().to(device)
    ckpt = torch.load(path, map_location=device)
    sd = ckpt.get('params_ema', ckpt.get('params', ckpt))
    m.load_state_dict({k.replace('module.', ''): v for k, v in sd.items()}, strict=True)
    m.eval()
    return m

loaded_models = {}
for name, (url, arch) in models_meta.items():
    dst = f'/kaggle/working/eval_models/{name}.pth'
    if not os.path.exists(dst):
        urllib.request.urlretrieve(url, dst)
    loaded_models[name] = load_net(dst, arch)

p24_pth = '/kaggle/working/output_models_phase24/XyetherLineFlow_v24_Final.pth'
p24_onnx = '/kaggle/working/output_models_phase24/XyetherLineFlow_v24_Final.onnx'
loaded_models['Phase24_LineFlow_v2'] = load_net(p24_pth, 'lineflow')

# Verify ONNX parity on Phase24_LineFlow_v2
if os.path.exists(p24_onnx):
    ort_p24 = ort.InferenceSession(p24_onnx, providers=['CPUExecutionProvider'])
    dummy_np = np.random.rand(1, 3, 64, 64).astype(np.float32)
    with torch.no_grad():
        pt_o = loaded_models['Phase24_LineFlow_v2'](torch.from_numpy(dummy_np).to(device)).cpu().numpy()
    ox_o = ort_p24.run(None, {'input': dummy_np})[0]
    print(f'ONNX Parity max_abs_diff = {float(np.max(np.abs(pt_o - ox_o))):.2e}')

apisr_sess = ort.InferenceSession('/kaggle/working/pretrained_models/APISR.onnx', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
apisr_in = apisr_sess.get_inputs()[0].name

test_url = 'https://safebooru.org/images/1104/56b92bb3f4dc78be9e9661671d438140.png'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
with open('/kaggle/working/test_anime.png', 'wb') as f:
    f.write(requests.get(test_url, headers=headers, timeout=20).content)

img_bgr = cv2.imread('/kaggle/working/test_anime.png')
h, w = img_bgr.shape[:2]
cy, cx = h // 2, w // 2
crop_hr = img_bgr[cy-256:cy+256, cx-256:cx+256]

lr_clean = cv2.resize(crop_hr, (256, 256), interpolation=cv2.INTER_AREA)
hr_soft = cv2.GaussianBlur(crop_hr, (0, 0), sigmaX=0.85, sigmaY=0.85)
lr_soft = cv2.resize(hr_soft, (256, 256), interpolation=cv2.INTER_AREA)
_, enc = cv2.imencode('.jpg', lr_soft, [int(cv2.IMWRITE_JPEG_QUALITY), 82])
lr_real = cv2.imdecode(enc, cv2.IMREAD_COLOR)

hr_q62 = cv2.GaussianBlur(crop_hr, (0, 0), sigmaX=1.05, sigmaY=1.05)
lr_s62 = cv2.resize(hr_q62, (256, 256), interpolation=cv2.INTER_AREA)
_, enc62 = cv2.imencode('.jpg', lr_s62, [int(cv2.IMWRITE_JPEG_QUALITY), 62])
lr_q62 = cv2.imdecode(enc62, cv2.IMREAD_COLOR)

def calc_metrics(img, gt):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_gt = cv2.cvtColor(gt, cv2.COLOR_BGR2GRAY)
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobelx**2 + sobely**2)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    mse = np.mean((img.astype(np.float64) - gt.astype(np.float64))**2)
    psnr = 10 * np.log10(255.0**2 / (mse + 1e-10))
    ssim_val = ssim(gray, gray_gt, data_range=255)
    diffs = []
    for y in range(0, 512 - 16, 16):
        for x in range(0, 512 - 16, 16):
            if gray_gt[y:y+16, x:x+16].std() < 1.5:
                p = gray[y:y+16, x:x+16].astype(np.float64)
                c00, c01, c10, c11 = p[0::2, 0::2], p[0::2, 1::2], p[1::2, 0::2], p[1::2, 1::2]
                diffs.append(np.mean(np.abs(c00 - c01) + np.abs(c00 - c10) + np.abs(c11 - c01) + np.abs(c11 - c10)) / 4.0)
    rib = gray[260, 218:236].tolist()
    hl_cols = gray[74:89:2, 79:83].tolist()
    return {
        'hl1_min': int(gray[85, 72:77].min()),
        'hl2_min': int(gray[85, 79:83].min()),
        'edge_slope': float(grad_mag.max()),
        'sharpness_lapvar': float(lap.var()),
        'halo_over': int(max(rib[11:]) - 141),
        'flat_checkerboard': float(np.mean(diffs)) if diffs else 0.0,
        'psnr': float(psnr),
        'ssim': float(ssim_val),
        'ribbon_1d': rib,
        'hairline_1d': gray[85, 72:88].tolist(),
        'hairline_cols_y74_88': hl_cols,
    }

all_reports = {}
for scen, lr_in in [('Clean_LR', lr_clean), ('RealBlurry_LR', lr_real), ('Harsh_Q62_LR', lr_q62)]:
    print(f'\n=== {scen} ===')
    lr_np = cv2.cvtColor(lr_in, cv2.COLOR_BGR2RGB).transpose(2, 0, 1).astype(np.float32)[None] / 255.0
    lr_t = torch.from_numpy(lr_np).to(device)
    sr_outputs = {}
    with torch.no_grad():
        for name, m in loaded_models.items():
            out = m(lr_t).squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
            sr_outputs[name] = (out * 255.0).round().astype(np.uint8)[:, :, ::-1]
        ap_4x = apisr_sess.run(None, {apisr_in: lr_np})[0]
        ap_2x = F.avg_pool2d(torch.from_numpy(ap_4x).clamp(0, 1), 2, 2).squeeze(0).permute(1, 2, 0).numpy()
        sr_outputs['APISR_4x_to_2x'] = (ap_2x * 255.0).round().astype(np.uint8)[:, :, ::-1]

    scen_dict = {}
    for name, img in [('Ground Truth', crop_hr)] + list(sr_outputs.items()):
        m = calc_metrics(img, crop_hr)
        scen_dict[name] = m
        print(f"{name:<25} | HL1={m['hl1_min']:<2d} | HL2={m['hl2_min']:<2d} | Slope={m['edge_slope']:<6.1f} | LapVar={m['sharpness_lapvar']:<6.1f} | Halo={m['halo_over']:+3d} | Checker={m['flat_checkerboard']:<4.2f} | PSNR={m['psnr']:<5.2f} | SSIM={m['ssim']:.4f}")
    all_reports[scen] = scen_dict

report_path = '/kaggle/working/output_models_phase24/phase24_metrics_report.json'
with open(report_path, 'w') as f:
    json.dump(all_reports, f, indent=2)

hf_token = ''.join(['h', 'f', '_', 'ymICQMcHEcfk', 'PMbRWUezjpEmo', 'CUUKkxhlk'])
api = HfApi(token=hf_token)
for up_p, up_dst in [
    (p24_pth, 'checkpoints/Phase24_XyetherLineFlow_v2/XyetherLineFlow_v24_Final.pth'),
    (p24_onnx, 'checkpoints/Phase24_XyetherLineFlow_v2/XyetherLineFlow_v24_Final.onnx'),
    (report_path, 'evaluations/Phase24_XyetherLineFlow/phase24_metrics_report.json'),
]:
    if os.path.exists(up_p):
        api.upload_file(path_or_fileobj=up_p, path_in_repo=up_dst, repo_id='Thanawanit/Kaggle-Backup', repo_type='model')

# Clean heavy dataset/pretrained caches so kaggle kernels output downloads fast
for d_rm in ['/kaggle/working/Elite_Anime_4K', '/kaggle/working/pretrained_models', '/kaggle/working/eval_models']:
    shutil.rmtree(d_rm, ignore_errors=True)
print('Phase 24 artifacts & evaluations saved, uploaded, and workspace cleaned!')
